# Google Colab Notebook

Environment setup

In [ ]:
# Clone the repository
!git clone https://github.com/nv-tlabs/nksr.git

# Move into the directory
%cd nksr

# Install dependencies
!pip install -r requirements.txt

# Build and install the NKSR package
!pip install --no-build-isolation package/

In [ ]:
!pip install open3d
!pip install -U "python-pycg[all]"

Verify Installation

In [ ]:
import nksr
import torch

print(f"NKSR Version: {nksr.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

Main

In [ ]:
from google.colab import files
import open3d as o3d
import numpy as np
import trimesh
import torch
import nksr
import os



# Upload PCD
print("\n- Upload a point cloud:")
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"-> Successfully loaded: {filename}")

# Load the file
pcd       = trimesh.load(filename)
points_np = pcd.vertices



# NKSR strictly requires normals...
print('\n- Estimating normals...')

# Convert to Open3D
o3d_pcd        = o3d.geometry.PointCloud()
o3d_pcd.points = o3d.utility.Vector3dVector(np.asarray(pcd.vertices))

# Calculate and orient the normals using Open3D
o3d_pcd.estimate_normals(search_param = o3d.geometry.KDTreeSearchParamKNN(knn = 120))
o3d_pcd.orient_normals_consistent_tangent_plane(k = 100)

normals_np = np.asarray(o3d_pcd.normals)

print(f'-> Successfully estimated {len(normals_np)} normals.\n')



# Convert to PyTorch tensors and move to GPU
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
points  = torch.tensor(np.array(points_np),  dtype = torch.float32).to(device)
normals = torch.tensor(np.array(normals_np), dtype = torch.float32).to(device)



# Initialize and Run NKSR
reconstructor = nksr.Reconstructor(device)
print(f"- Reconstructing from {len(points)} points...")
field         = reconstructor.reconstruct(points, normals)
mesh_result   = field.extract_dual_mesh()

# Convert back to NumPy and save
vertices = mesh_result.v.cpu().numpy()
faces    = mesh_result.f.cpu().numpy()



# Dynamically name the output file based on input...
base_name       = os.path.splitext(filename)[0]
output_filename = f"reconstructed_{base_name}.ply"

exportable_mesh = trimesh.Trimesh(vertices = vertices, faces = faces)
exportable_mesh.export(output_filename)

print(f"-> Success! Preparing to download '{output_filename}'...")

# AUTOMATIC DOWNLOAD
files.download(output_filename)